# Exploratory Data Analysis — Fake Laugh Detector

Use this notebook to explore your dataset: waveform plots, spectrograms,
and feature distributions between REAL and FAKE laughter classes.

In [ ]:
import sys
sys.path.append('../src')

import glob
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from feature_extraction import extract_features, FEATURE_NAMES

In [ ]:
# Load one example clip from each class and plot waveform + spectrogram
real_files = glob.glob('../data/real/*.wav')
fake_files = glob.glob('../data/fake/*.wav')
print(f'Real clips: {len(real_files)}, Fake clips: {len(fake_files)}')

In [ ]:
if real_files and fake_files:
    fig, axes = plt.subplots(2, 2, figsize=(12, 6))
    for i, (label, f) in enumerate([('REAL', real_files[0]), ('FAKE', fake_files[0])]):
        y, sr = librosa.load(f)
        librosa.display.waveshow(y, sr=sr, ax=axes[0, i])
        axes[0, i].set_title(f'{label} waveform')

        S = librosa.feature.melspectrogram(y=y, sr=sr)
        S_dB = librosa.power_to_db(S, ref=np.max)
        img = librosa.display.specshow(S_dB, sr=sr, ax=axes[1, i])
        axes[1, i].set_title(f'{label} mel-spectrogram')
    plt.tight_layout()
    plt.show()

In [ ]:
# Build a feature dataframe across the whole dataset for distribution plots
rows = []
for f in real_files:
    rows.append(list(extract_features(f)) + ['REAL'])
for f in fake_files:
    rows.append(list(extract_features(f)) + ['FAKE'])

df = pd.DataFrame(rows, columns=FEATURE_NAMES + ['label'])
df.describe()

In [ ]:
import seaborn as sns

for feat in ['pitch_mean', 'pitch_std', 'tempo', 'rms_std']:
    if feat in df.columns:
        plt.figure(figsize=(5,3))
        sns.boxplot(data=df, x='label', y=feat)
        plt.title(feat)
        plt.show()